# Flow matching for simple labelled distributions

In this notebook we will show how flow matching and rectified flow can be used to sample data from a toy 2D distribution with labels.

In [ ]:
%load_ext autoreload
%autoreload 2

import data  # Auxiliary module for generating toy datasets
import models  # Auxiliary module for neural network architecture and training
import plotting  # Auxiliary module for visualizations

## Load toy labelled data

Generate target distribution data and original noisy data (a standard gaussian)

In [ ]:
import numpy as np

target_data, target_labels = data.generate_toy_data("two_gaussians_supervised", n=1000)
unique_labels = set(target_labels)

num_couplings= 100000
source_data = np.random.randn(num_couplings, 2)

plotting.plot_distributions(source_data, target_data, target_labels=target_labels)

## Basic flow matching

Let's sample independent couplings between source and target data points

In [ ]:
couplings = models.sample_independent_couplings(source_data, target_data, num_couplings=num_couplings, target_labels=target_labels)
plotting.plot_distributions(source_data, target_data, target_labels=target_labels, couplings=couplings)

Train a simple MLP as the velocity field estimation network.

In [ ]:
network = models.train_flow_model(couplings, verbose=True)

Visualize the velocity fields we have learned: unconditional and conditional for both classes.

In [ ]:
plotting.plot_class_conditional_velocity_fields(network, source_data, target_data, target_labels=target_labels)

Now that we have the velocity field, we can use a integration method such as Euler's method to run across the flow (for now with no labels)

In [ ]:
source_data = np.random.randn(1000, 2)
trajectories = models.compute_trajectories(network, source_data, label=None)
plotting.animate_trajectories(trajectories, target_data=target_data)

We can also generate trajectories conditioned on the class labels to lead the generation process towards a particular class

In [ ]:
class_trajectories = {
    label: models.compute_trajectories(network, np.random.randn(500, 2), label=label)
    for label in unique_labels
}
plotting.animate_trajectories(class_trajectories, target_data=target_data)

## Rectified flows

Rectification can also be applied to conditioned flows. To do so, we just need to use the trajectories generated for the different classes, and use those to produce new couplings.

In [ ]:
# Generate trajectories for each class and collect the final points and labels

n_per_class = num_couplings // len(unique_labels)
source_data, synthetic_data, synthetic_labels = [], [], []
for label in unique_labels:
    class_source_data = np.random.randn(n_per_class, 2)
    class_trajectories = models.compute_trajectories(network, class_source_data, label=label)
    source_data.append(class_source_data)
    synthetic_data.append([traj[-1][1] for traj in class_trajectories])
    synthetic_labels.append([label] * n_per_class)
source_data = np.concatenate(source_data, axis=0)
synthetic_data = np.concatenate(synthetic_data, axis=0)
synthetic_labels = np.concatenate(synthetic_labels, axis=0)

# Shuffle the couplings to mix labels along the dataset
couplings = [(src, tgt, label) for src, tgt, label in zip(source_data, synthetic_data, synthetic_labels)]
np.random.shuffle(couplings)

plotting.plot_distributions(source_data, synthetic_data, couplings=couplings, target_labels=synthetic_labels)

In [ ]:
rectified_network = models.train_flow_model(couplings, verbose=True)
plotting.plot_class_conditional_velocity_fields(rectified_network, source_data, target_data, target_labels=target_labels)

The trajectories should be straigther in the new fields

In [ ]:
source_data = np.random.randn(1000, 2)
rectified_trajectories = models.compute_trajectories(rectified_network, source_data, label=None)
plotting.animate_trajectories(rectified_trajectories, target_data=target_data)

In [ ]:
rectified_class_trajectories = {
    label: models.compute_trajectories(rectified_network, np.random.randn(500, 2), label=label)
    for label in unique_labels
}
plotting.animate_trajectories(rectified_class_trajectories, target_data=target_data)

In [ ]:
plotting.plot_class_conditioned_generated_data_comparison(target_data, rectified_class_trajectories, target_labels)